# ラベルと特徴量エンジニアリング

学習目標:

1. BigQuery を使用して予測のための時系列特徴量とラベルを構築する方法を学ぶ
2. 特徴量を可視化および探索する方法を学ぶ
3. モデリング結果を向上させるための効果的なスケーリングおよび正規化手法を学ぶ

データの探索が完了したので、モデルを構築できるように特徴量の作成を始めましょう。

<h3><font color="#4885ed">特徴量エンジニアリング</font> </h3>

`price_history` テーブルを使用すると、特定の銘柄の過去のパフォーマンスを確認して、将来の株価を予測することができます。このノートブックでは、このテーブルのクレンジングと特徴量の作成に焦点を当てます。

時系列データから特徴量を作成するには、一般的に 2 つの異なるアプローチがあります。

**1 つ目のアプローチ** は、時系列データを「過去 1 か月の最低価格（min_price_over_past_month）」や「過去 30 日間の指数移動平均（exp_moving_avg_past_30_days）」のような「静的」な特徴量に集約することです。このアプローチでは、ディープ ニューラル ネットワークや、より「従来型」の ML モデルを使用してトレーニングできます。集約後には、実質的にすべての順序情報が削除されていることに注目してください。この仮定は実際にはうまく機能することがよくあります。

**2 つ目のアプローチ** は、データの順序性を保持し、リカレント ニューラル ネットワークなどのシーケンシャル モデルを使用することです。このアプローチには、通常、特徴量エンジニアリングの必要性が少ないという利点があります。ただし、シーケンシャル モデルのトレーニングには通常より長い時間がかかります。

このノートブックでは、特徴量を構築するとともに、順序付けられた時系列データのローリング ウィンドウも作成します。

<h3><font color="#4885ed">ラベル エンジニアリング</font> </h3>

ここでは株価が上がるか下がるかを予測しようとしています。これを行うには、将来の時点を確認し、それをラベルとして使用することで、ラベルを「エンジニアリング（設計）」する必要があります。これを行うために、BigQuery の [`LAG`](https://cloud.google.com/bigquery/docs/reference/standard-sql/functions-and-operators#lag) 関数を使用します。視覚的には次のようになります。

![](https://github.com/GoogleCloudPlatform/training-data-analyst/blob/master/blogs/gcp_forecasting/rolling_window.gif?raw=true)

## ライブラリのインポートとセットアップ

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

In [ ]:
import pandas as pd
from google.cloud import bigquery
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

%load_ext google.cloud.bigquery

bq = bigquery.Client(project=PROJECT)

In [ ]:
# SQL クエリ内で Python 変数を簡単に使用できるようにします。
@register_cell_magic("with_globals")
def with_globals(line, cell):
    contents = cell.format(**globals())
    if "print" in line:
        print(contents)
    get_ipython().run_cell(contents)

In [ ]:
def create_dataset():
    dataset = bigquery.Dataset(bq.dataset("stock_market"))
    try:
        bq.create_dataset(dataset)  # Will fail if dataset already exists.
        print("Dataset created")
    except:
        print("Dataset already exists")


create_dataset()

## 時系列特徴量の作成と市場動向に基づくラベルの決定

### ベーステーブルのサマリー

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
SELECT count(*) as cnt
FROM `stock_src.price_history`

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
SELECT count(*) as cnt
FROM `stock_src.snp500`

### ラベル エンジニアリング

最終的には、各日に対して 1 つのラベルを取得する必要があります。ラベルは {`down`, `stay`, `up`} の 3 つの値を持ちます。`down` と `up` は、正規化された価格（詳細は後述）がそれぞれ 1% 以上下落したか、1% 以上上昇したかを示します。`stay` は株価が 1% 以内にとどまったことを示します。

ステップは以下のとおりです。

1. 終値と始値の比較
2. 分析関数を使用した価格特徴量の計算
3. 正規化された価格変動率（%）の計算
4. S&P 500 テーブルとの結合
5. ラベル（`up`, `down`, `stay`）の作成

<h3><font color="#4885ed">終値と始値の比較</font> </h3>

各行について、[`LAG`](https://cloud.google.com/bigquery/docs/reference/legacy-sql#lag) 関数を使用して前日の終値と翌日の始値を取得します。ここでは「翌日の終値 - 当日の終値」を求めます。

#### 翌日の終値を取得するためのシフト

**学習目標 1**

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
CREATE OR REPLACE TABLE `stock_market.price_history_delta`
AS
(
WITH shifted_price AS
  (
  SELECT *,
  (LAG(close, 1) OVER (PARTITION BY symbol order by Date DESC)) AS tomorrow_close
  FROM `stock_src.price_history`
  WHERE Close > 0
  )
SELECT a.*,
(tomorrow_close - Close) AS tomo_close_m_close
FROM shifted_price a
)

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
SELECT *
FROM stock_market.price_history_delta
ORDER by Date
LIMIT 100

株式市場は平均して上昇傾向にあります。

**学習目標 2**

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT
AVG(close) AS avg_close,
AVG(tomorrow_close) AS avg_tomo_close,
AVG(tomorrow_close) - AVG(close) AS avg_change,
COUNT(*) cnt
FROM
stock_market.price_history_delta

### 時系列特徴量の追加

<h3><font color="#4885ed">分析関数を使用した価格特徴量の計算</font> </h3>

さらに、最小値（min）、最大値（max）、平均値（mean）、標準偏差（std）を使用して時系列特徴量も構築します（他に使用できる関数は思いつきますか？）。これを行うために、BigQuery の [分析関数（analytic functions）]()（ウィンドウ関数とも呼ばれます）を使用しましょう。
```
分析関数は、一連の行にわたって集計値を計算する関数です。一連の行に対して単一の集計値を返す集約関数とは異なり、分析関数は入力行のグループに対して関数を計算することで、各行に対して 1 つの値を返します。
```
 `AVG` 分析関数を使用すると、特定の銘柄の過去 1 週間（5 営業日）の平均終値を計算できます。
```python
 (AVG(close) OVER (PARTITION BY symbol
  ORDER BY Date
  ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING)) / close
  AS close_avg_prior_5_days
```

**学習目標 1**

In [ ]:
def get_window_fxn(agg_fxn, n_days):
    """時系列特徴量を生成します。

    例: 過去 5 日間の価格の平均を計算する。"""
    SCALE_VALUE = "close"
    sql = """
    ({agg_fxn}(close) OVER (PARTITION BY symbol
                      ORDER BY Date
                      ROWS BETWEEN {n_days} PRECEDING AND 1 PRECEDING))/{scale}
                      AS close_{agg_fxn}_prior_{n_days}_days""".format(
        agg_fxn=agg_fxn, n_days=n_days, scale=SCALE_VALUE
    )
    return sql


WEEK = 5
MONTH = 20
YEAR = 52 * 5

agg_funcs = ("MIN", "MAX", "AVG", "STDDEV")
lookbacks = (WEEK, MONTH, YEAR)
sqls = []

for fxn in agg_funcs:
    for lookback in lookbacks:
        sqls.append(get_window_fxn(fxn, lookback))
time_series_features_sql = ",".join(sqls)  # SQL string.


def preview_query():
    print(time_series_features_sql[0:1000])


preview_query()

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.price_features_delta
AS
SELECT * 
FROM 
    (SELECT *,
    {time_series_features_sql},
    -- 生の時系列値も取得。RNN モデルで役立ちます。
    (ARRAY_AGG(close) OVER (PARTITION BY symbol
                    ORDER BY Date
                    ROWS BETWEEN 260 PRECEDING AND 1 PRECEDING))
                    AS close_values_prior_260,
    ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY Date) AS days_on_market
    FROM stock_market.price_history_delta)
WHERE days_on_market > {YEAR}

In [ ]:
%%bigquery --project {PROJECT}
SELECT *
FROM stock_market.price_features_delta 
ORDER BY symbol, Date
LIMIT 10

#### 変動率を計算し、S&P 指数の価格と自己結合する

S&P 指数（GSPC）の価格変動も計算します。これは、正規化された変動率を計算できるようにするためです。

<h3><font color="#4885ed">正規化された価格変動率（%）の計算</font> </h3>

ラベルを作成する前に、S&P 500 指数を使用して価格変動を正規化する必要があります。S&P インデックス ファンドを使用した正規化は、株式の将来の価格が市場全体の影響によるものではないことを確認するのに役立ちます。正規化により、株式市場のパフォーマンスに寄与する要因を分離できます。

同時期の S&P 500 指数のスケーリングされた変動を差し引く正規化スキームを使用しましょう。

Python での例:
```python
# 計算例
scaled_change = (50.59 - 50.69) / 50.69
scaled_s_p = (939.38 - 930.09) / 930.09
normalized_change = scaled_change - scaled_s_p
assert normalized_change == ~1.2%
```

In [ ]:
scaled_change = (50.59 - 50.69) / 50.69
scaled_s_p = (939.38 - 930.09) / 930.09
normalized_change = scaled_change - scaled_s_p
print(
    """
scaled change: {:2.3f}
scaled_s_p: {:2.3f}
normalized_change: {:2.3f}
""".format(
        scaled_change, scaled_s_p, normalized_change
    )
)

### 正規化された価格変動の計算（上記を参照）

スケーリングされた価格変動（tomorrow_close / close）を [gspc](https://en.wikipedia.org/wiki/S%26P_500_Index) シンボル（S&P 指数のシンボル）と結合しましょう。その後、上記で説明したスキームを使用して正規化できます。

**学習目標 3**

In [ ]:
snp500_index = "gspc"

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.price_features_norm_per_change
AS
WITH 
all_percent_changes AS
    (
    SELECT *, (tomo_close_m_close / Close) AS scaled_change
    FROM `stock_market.price_features_delta`
    ),
s_p_changes AS
    (SELECT
     scaled_change AS s_p_scaled_change,
     date
     FROM all_percent_changes
     WHERE symbol="{snp500_index}")

SELECT all_percent_changes.*,
       s_p_scaled_change,
       (scaled_change - s_p_scaled_change) 
           AS normalized_change          
FROM 
    all_percent_changes LEFT JOIN s_p_changes
    --# すべての行に S&P の変動を追加
    ON all_percent_changes.date = s_p_changes.date

#### 結果の確認

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
SELECT *
FROM stock_market.price_features_norm_per_change
LIMIT 10

In [ ]:
df.head()

<h3><font color="#4885ed">S&P 500 テーブルとの結合とラベルの作成: {`up`, `down`, `stay`}</font> </h3>

テーブルを S&P 500 のリストと結合します。これにより、分析対象を S&P 500 構成企業のみに限定できます。

最後にラベルを作成します。次の SQL ステートメントで実行できます。

```sql
  CASE WHEN normalized_change < -0.01 THEN 'DOWN'
       WHEN normalized_change > 0.01 THEN 'UP'
       ELSE 'STAY'
  END
```

**学習目標 1**

In [ ]:
down_thresh = -0.01
up_thresh = 0.01

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.percent_change_sp500
AS
SELECT *,
    CASE WHEN normalized_change < {down_thresh} THEN 'DOWN'
       WHEN normalized_change > {up_thresh} THEN 'UP'
       ELSE 'STAY'
    END AS direction
FROM stock_market.price_features_norm_per_change features
INNER JOIN `stock_src.snp500`
USING (symbol)

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT direction, COUNT(*) as cnt
FROM stock_market.percent_change_sp500
GROUP BY direction

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
SELECT *
FROM stock_market.percent_change_sp500
LIMIT 20

In [ ]:
df.columns

データセットは依然として非常に大きく、大半の日は相場が「`STAY`」にとどまっています。各企業によって[一株当たり利益](https://en.wikipedia.org/wiki/Earnings_per_share)（EPS）情報が発表された日付に焦点を当てて分析してみましょう。EPS データには、surprise、reported_EPS、consensus_EPS という 3 つの主要な列があります。

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT *
FROM `stock_src.eps`
LIMIT 10

surprise 列は、予想（アナリストによるコンセンサス予想 EPS）と発表された EPS の差を示します。このテーブルを作成した派生テーブルと結合して、決算発表期間中の分析に焦点を当てることができます。

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.eps_percent_change_sp500
AS
SELECT a.*, b.consensus_EPS, b.reported_EPS, b.surprise
FROM stock_market.percent_change_sp500 a
INNER JOIN `stock_src.eps` b
ON a.Date = b.date
AND a.symbol = b.symbol

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT *
FROM stock_market.eps_percent_change_sp500
LIMIT 5

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT direction, COUNT(*) as cnt
FROM stock_market.eps_percent_change_sp500
GROUP BY direction

## 特徴量の探索

企業の株価の直近の動きを作成できたので、特徴量を可視化してみましょう。これにより、データをより深く理解でき、計算中に犯した可能性のある誤りを発見できる場合があります。

補足として、予測対象の日の 1 週間前、1 か月前、1 年前のスケーリングされた価格を計算しました。

特徴量を集約するための再利用可能な関数を作成しましょう。

**学習目標 2**

In [ ]:
def get_aggregate_stats(field, round_digit=2):
    """SELECT ... GROUP BY field を実行し、最も近い桁に四捨五入します。"""
    df = bq.query(
        """
    SELECT {field}, COUNT(*) as cnt
    FROM
    (SELECT ROUND({field}, {round_digit}) AS {field}
    FROM stock_market.eps_percent_change_sp500) rounded_field
    GROUP BY {field}
    ORDER BY {field}""".format(
            field=field, round_digit=round_digit, PROJECT=PROJECT
        )
    ).to_dataframe()
    return df.dropna()

In [ ]:
field = "close_AVG_prior_260_days"
CLIP_MIN, CLIP_MAX = 0.1, 4.0
df = get_aggregate_stats(field)
values = df[field].clip(CLIP_MIN, CLIP_MAX)
counts = 100 * df["cnt"] / df["cnt"].sum()  # パーセンテージ
ax = values.hist(weights=counts, bins=30, figsize=(10, 5))
ax.set(xlabel=field, ylabel="%");

In [ ]:
field = "normalized_change"
CLIP_MIN, CLIP_MAX = -0.1, 0.1
df = get_aggregate_stats(field, round_digit=3)
values = df[field].clip(CLIP_MIN, CLIP_MAX)
counts = 100 * df["cnt"] / df["cnt"].sum()  # パーセンテージ
ax = values.hist(weights=counts, bins=50, figsize=(10, 5))
ax.set(xlabel=field, ylabel="%");

曜日別、月別などの結果を見てみましょう。

In [ ]:
VALID_GROUPBY_KEYS = (
    "DAYOFWEEK",
    "DAY",
    "DAYOFYEAR",
    "WEEK",
    "MONTH",
    "QUARTER",
    "YEAR",
)
DOW_MAPPING = {
    1: "Sun",
    2: "Mon",
    3: "Tues",
    4: "Wed",
    5: "Thur",
    6: "Fri",
    7: "Sun",
}


def groupby_datetime(groupby_key, field):
    if groupby_key not in VALID_GROUPBY_KEYS:
        raise Exception("Please use a valid groupby_key.")
    sql = """
    SELECT {groupby_key}, AVG({field}) as avg_{field}
    FROM
    (SELECT {field},
     EXTRACT({groupby_key} FROM date) AS {groupby_key}
    FROM stock_market.eps_percent_change_sp500) foo
    GROUP BY {groupby_key}
    ORDER BY {groupby_key} DESC""".format(
        groupby_key=groupby_key, field=field, PROJECT=PROJECT
    )
    print(sql)
    df = bq.query(sql).to_dataframe()
    if groupby_key == "DAYOFWEEK":
        df.DAYOFWEEK = df.DAYOFWEEK.map(DOW_MAPPING)
    return df.set_index(groupby_key).dropna()

In [ ]:
field = "normalized_change"

df = groupby_datetime("DAYOFWEEK", field)
ax = df.plot(kind="barh", color="orange", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "close"
df = groupby_datetime("DAYOFWEEK", field)
ax = df.plot(kind="barh", color="orange", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "normalized_change"
df = groupby_datetime("MONTH", field)
ax = df.plot(kind="barh", color="blue", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "normalized_change"
df = groupby_datetime("QUARTER", field)
ax = df.plot(kind="barh", color="green", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "close"
df = groupby_datetime("YEAR", field)
ax = df.plot(kind="line", color="purple", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "normalized_change"
df = groupby_datetime("YEAR", field)
ax = df.plot(kind="line", color="purple", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

ボーナス: 作成した特徴量はラベル `direction` とどのように相関しているでしょうか。いくつか可視化を作成してみてください。最も重要な特徴量は何でしょうか。これを可視化し、[`CORR`](https://cloud.google.com/bigquery/docs/reference/standard-sql/statistical_aggregate_functions) 関数を使用して統計的に分析してみてください。

Copyright 2019 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License